In [37]:
import json
from dateutil.parser import parse
import numpy as np
import pandas as pd
import seaborn as sns

#https://blog.trakt.tv/year-in-review-5c6ac98f0d3c

In [38]:
history_file = '../content/watches/summary/.history.json'
year = 2021

def update_watched_at(item):
  item['watched_at'] = parse(item['watched_at'])
  return item

history = json.load(open(history_file))
history = map(update_watched_at, history)
history = filter(lambda item: item['watched_at'].year == year, history)
history = list(history)

In [39]:
def convert_item(item):
  new_item = { **item }
  del new_item['action']
  if 'episode' in new_item: del new_item['episode']
  if 'show' in new_item: del new_item['show']
  if 'movie' in new_item: del new_item['movie']

  if item['type'] == 'episode':
    new_item['runtime'] = int(item['show']['runtime'])
    new_item['title'] = item['show']['title']
    new_item['genres'] = list(map(lambda x: x.title(), item['show']['genres']))
    new_item['episode'] = int(item['episode']['number'])
    new_item['season'] = int(item['episode']['season'])
  else:
    new_item['runtime'] = int(item['movie']['runtime'])
    new_item['title'] = item['movie']['title']
    new_item['genres'] = list(map(lambda x: x.title(), item['movie']['genres']))
  
  return new_item

df = pd.DataFrame(map(convert_item, history))
df = df.sort_values('watched_at')

df['month'] = df.watched_at.dt.month
df['day'] = df.watched_at.dt.day

df

,id,watched_at,type,runtime,title,genres,episode,season,month,day
912,7065954732,2021-01-02 08:39:22+00:00,episode,25,The Unlisted,"[Family, Fantasy, Science-Fiction]",6.0,1.0,1,2
911,7065954741,2021-01-02 08:39:24+00:00,episode,25,The Unlisted,"[Family, Fantasy, Science-Fiction]",7.0,1.0,1,2
910,7065954822,2021-01-02 08:39:28+00:00,episode,25,The Unlisted,"[Family, Fantasy, Science-Fiction]",8.0,1.0,1,2
909,7065954839,2021-01-02 08:39:29+00:00,episode,25,The Unlisted,"[Family, Fantasy, Science-Fiction]",9.0,1.0,1,2
908,7065954851,2021-01-02 08:39:33+00:00,episode,25,The Unlisted,"[Family, Fantasy, Science-Fiction]",10.0,1.0,1,2
...,...,...,...,...,...,...,...,...,...,...
4,8042696204,2021-12-31 14:35:42+00:00,episode,45,Travelers,"[Drama, Fantasy, Science-Fiction]",9.0,1.0,12,31
3,8042696249,2021-12-31 14:35:48+00:00,episode,45,Travelers,"[Drama, Fantasy, Science-Fiction]",10.0,1.0,12,31
2,8042739443,2021-12-31 15:16:37+00:00,episode,45,Travelers,"[Drama, Fantasy, Science-Fiction]",11.0,1.0,12,31
1,8043008626,2021-12-31 18:32:07+00:00,episode,45,Travelers,"[Drama, Fantasy, Science-Fiction]",12.0,1.0,12,31


In [40]:
def genre_count(df):
  genres = df['genres'].explode().value_counts(sort=True, ascending=False).rename_axis('genre').reset_index(name='count')
  genres['percentage'] = (genres['count'] / genres['count'].sum()).mul(100).round(1)
  return genres

In [41]:
# Genre Count By Number of Movies
md = genre_count(df[df['type'] == 'movie'].drop_duplicates(subset=['title'], keep='last')).set_index('genre').to_markdown()
print(md)

| genre           |   count |   percentage |
|:----------------|--------:|-------------:|
| Comedy          |      32 |         24.1 |
| Romance         |      24 |         18   |
| Family          |      16 |         12   |
| Fantasy         |      12 |          9   |
| Drama           |      12 |          9   |
| Adventure       |       9 |          6.8 |
| Animation       |       8 |          6   |
| Action          |       5 |          3.8 |
| Science-Fiction |       3 |          2.3 |
| Holiday         |       3 |          2.3 |
| Documentary     |       2 |          1.5 |
| Superhero       |       1 |          0.8 |
| Musical         |       1 |          0.8 |
| Horror          |       1 |          0.8 |
| Mystery         |       1 |          0.8 |
| Thriller        |       1 |          0.8 |
| Crime           |       1 |          0.8 |
| History         |       1 |          0.8 |


In [42]:
# Genre Count By Number of Shows
md = genre_count(df[df['type'] == 'episode'].drop_duplicates(subset=['title'], keep='last')).set_index('genre').to_markdown()
print(md)

| genre           |   count |   percentage |
|:----------------|--------:|-------------:|
| Drama           |      31 |         23.8 |
| Science-Fiction |      19 |         14.6 |
| Fantasy         |      19 |         14.6 |
| Comedy          |      14 |         10.8 |
| Mystery         |      13 |         10   |
| Reality         |       6 |          4.6 |
| Action          |       5 |          3.8 |
| Adventure       |       5 |          3.8 |
| Crime           |       4 |          3.1 |
| Family          |       3 |          2.3 |
| Romance         |       2 |          1.5 |
| Thriller        |       2 |          1.5 |
| Documentary     |       2 |          1.5 |
| Animation       |       1 |          0.8 |
| Holiday         |       1 |          0.8 |
| Suspense        |       1 |          0.8 |
| Superhero       |       1 |          0.8 |
| Children        |       1 |          0.8 |


In [43]:
unique_shows = df[df['type'] == 'episode']['title'].unique()
print('Unique Shows', len(unique_shows), unique_shows)

Unique Shows 51 ['The Unlisted' 'Locke & Key' "RuPaul's Drag Race"
 "RuPaul's Drag Race: Untucked!" 'Greenhouse Academy' 'The Good Doctor'
 "RuPaul's Drag Race UK" 'Good Witch' 'Young Sheldon'
 'Fate: The Winx Saga' 'Snowpiercer' 'BONDiNG' 'The Good Place'
 'Kurzgesagt - In a Nutshell' 'Tiny Pretty Things' 'The Rain'
 'Shadowhunters' "RuPaul's Drag Race All Stars: Untucked!"
 "Heathrow: Britain's Busiest Airport" 'Good Girls' 'The Irregulars'
 'The Shannara Chronicles' "RuPaul's Drag Race Down Under" 'Elite'
 'Special' 'Young Royals' "The Handmaid's Tale" 'Love, Victor'
 'The A List' "RuPaul's Drag Race All Stars" 'Never Have I Ever'
 'Shadow and Bone' "Marvel's Iron Fist" 'Control Z' 'Ragnarok' 'Riverdale'
 'Sunset' 'Sex Education' 'Downton Abbey' 'Squid Game' 'Dark' 'Big Mouth'
 'Elves' 'Anne with an E' 'Dash & Lily' 'The Witcher' 'The Worst Witch'
 'Elite Short Stories: Patrick' 'Elite Short Stories: Samuel Omar'
 'Degrassi: Next Class' 'Travelers']


In [44]:
# Times Watched Each Show
df[df['type'] == 'episode']['title'].value_counts()

Shadowhunters                              56
The Worst Witch                            52
The Good Place                             50
Downton Abbey                              49
Good Witch                                 48
Greenhouse Academy                         40
Riverdale                                  35
Anne with an E                             27
Dark                                       26
Degrassi: Next Class                       24
Marvel's Iron Fist                         23
Young Sheldon                              22
The Good Doctor                            22
RuPaul's Drag Race UK                      20
Never Have I Ever                          20
Locke & Key                                20
The Shannara Chronicles                    20
Kurzgesagt - In a Nutshell                 18
RuPaul's Drag Race                         16
Sunset                                     16
Good Girls                                 14
The A List                        

In [45]:
unique_movies = df[df['type'] == 'movie']['title'].unique()
print('Unique Movies', len(unique_movies), unique_movies)

Unique Movies 49 ['Soul' 'Winx Club: The Secret of the Lost Kingdom' 'Thunder Force'
 'What Men Want' 'Love and Monsters' 'Moana' 'Until Porn Do Us Part'
 'Cryptopia: Bitcoin, Blockchains & the Future of the Internet'
 'The House of Flowers: The Movie' 'Good on Paper' 'Black Widow'
 'Resort to Love' 'Luca' 'The Kissing Booth' 'The Kissing Booth 2'
 'The Kissing Booth 3' "To All the Boys I've Loved Before"
 'To All the Boys: P.S. I Still Love You'
 'To All the Boys: Always and Forever' 'Sierra Burgess Is a Loser'
 "He's All That" 'Just Ask Him' 'Teens Like Phil' 'Tyler' 'Filip' 'Him'
 'Confessions of an Invisible Girl' 'Hereditary' 'In a Heartbeat'
 'Single All the Way' 'A Castle for Christmas' 'A Boy Called Christmas'
 'Afterlife of the Party' 'House of Gucci' 'Raya and the Last Dragon'
 'Love Hard' 'The Matrix' 'Ferdinand' 'The Princess Switch'
 'The Claus Family' 'David and the Elves' 'The Christmas Chronicles'
 'The Christmas Chronicles: Part Two'
 'The Princess Switch: Switched Aga

In [46]:
# Times Watched Each Movie
df[df['type'] == 'movie']['title'].value_counts()

Soul                                                            1
Him                                                             1
Hereditary                                                      1
In a Heartbeat                                                  1
Single All the Way                                              1
A Castle for Christmas                                          1
A Boy Called Christmas                                          1
Afterlife of the Party                                          1
House of Gucci                                                  1
Raya and the Last Dragon                                        1
Love Hard                                                       1
The Matrix                                                      1
Ferdinand                                                       1
The Princess Switch                                             1
The Claus Family                                                1
David and 

In [47]:
# First Watch
df.iloc[0]

id                                    7065954732
watched_at             2021-01-02 08:39:22+00:00
type                                     episode
runtime                                       25
title                               The Unlisted
genres        [Family, Fantasy, Science-Fiction]
episode                                      6.0
season                                       1.0
month                                          1
day                                            2
Name: 912, dtype: object

In [48]:
# Last Watch
df.iloc[-1]

id                                   8043174139
watched_at            2021-12-31 20:54:46+00:00
type                                    episode
runtime                                      45
title                                 Travelers
genres        [Drama, Fantasy, Science-Fiction]
episode                                     1.0
season                                      2.0
month                                        12
day                                          31
Name: 0, dtype: object

In [49]:
# Plot per weekday of shows   (average minutes per day)
# Plot per weekday of movies  (average minutes per day)
# Plot per month